# 03 — The media-mix model, fitted wrong on purpose

Two fits on the panel with the real Olist baseline. The **misspecified** arm leads,
because it is the situation an analyst is actually in: nobody knows the true
functional form. The **matched** arm is a control, and reporting only it would be
circular.

Priors are pymc-marketing's defaults, unchanged. This matters more than it sounds:
prior choice is where circularity gets into a recovery study without anyone
noticing, and a prior centred near the true ROI would produce excellent recovery
and prove nothing. The consequence is wide intervals, which is a finding rather
than a defect.

In [1]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [2]:
mmm = read_metric("mmm", METRICS)
print(mmm["truth_access"])
print()
for name, description in mmm["design"]["specifications"].items():
    print(f"{name:14s} {description}")

Both fits were written to disk before the ground truth was read. athar.truth.load_truth refuses until the artifact it is asked to score exists, so no estimate here can have been informed by the answer.

matched        DelayedAdstock + HillSaturation — the generating form, as a control
misspecified   GeometricAdstock + LogisticSaturation — cannot express the generating form


## Did the sampler work

A fit that fails its diagnostics is reported as failed rather than folded into a
result. The threshold is a divergence *rate* rather than a count, because a count
tightens as you sample more and is therefore not a rule.

In [3]:
show(pd.DataFrame([
    {"specification": name, **{k: v for k, v in fit["diagnostics"].items()
                                if k not in ("criteria", "criteria_strict")}}
    for name, fit in mmm["fits"].items()
]))

specification  divergence_rate  divergences  max_r_hat  min_ess_bulk  passed  passed_strict  post_warmup_draws
      matched          0.00125            5   1.010080        653.35   False          False               4000
 misspecified          0.00075            3   1.003461       1400.39    True          False               4000



## Recovery

Coverage leads. Whether the true value falls inside the interval is the property
the model actually claims, and it is checkable; a point estimate that lands close
on one draw is an anecdote.

In [4]:
for name, fit in mmm["fits"].items():
    rows = [{"channel": c, **{k: round(v, 4) if isinstance(v, float) else v
                              for k, v in entry.items()}}
            for c, entry in fit["average_roi"]["channels"].items()]
    show(pd.DataFrame(rows)[["channel", "true", "estimated_mean", "hdi_low", "hdi_high",
                             "covered", "relative_error"]],
         f"{name} — average ROI")
    print("  summary:", {k: round(v, 4) if isinstance(v, float) else v
                          for k, v in fit["average_roi"]["summary"].items()})
    print()

matched — average ROI
        channel  true  estimated_mean  hdi_low  hdi_high  covered  relative_error
   display_prog   0.9          9.2454   0.0002   24.2708     True          9.2726
   search_brand   1.6         15.2235   0.0001   43.6401     True          8.5147
search_nonbrand   2.8          8.4985   0.0000   24.5860     True          2.0352
    social_paid   2.1         10.2667   0.0001   30.8537     True          3.8889
      video_ctv   2.5         16.0643   0.0001   44.2645     True          5.4257

  summary: {'channels_covered': 5, 'channels_total': 5, 'coverage_rate': 1.0, 'hdi_prob': 0.89, 'mean_interval_width': 33.5229, 'median_absolute_relative_error': 5.4257}

misspecified — average ROI
        channel  true  estimated_mean  hdi_low  hdi_high  covered  relative_error
   display_prog   0.9          1.1341   0.0001    2.4706     True          0.2602
   search_brand   1.6          1.5859   0.0002    3.5438     True         -0.0088
search_nonbrand   2.8          0.6811   0

## Marginal ROI, which is what the budget decision needs

Scored separately, because a model can be respectable on average ROI and useless
on the slope — and the slope is the quantity notebook 08 allocates on.

In [5]:
for name, fit in mmm["fits"].items():
    summary = fit["marginal_roi"]["summary"]
    print(f"{name:14s} coverage {summary['coverage_rate']:.2f}  "
          f"median |rel err| {summary['median_absolute_relative_error']:.2f}  "
          f"mean interval width {summary['mean_interval_width']:.2f}")

matched        coverage 0.80  median |rel err| 0.35  mean interval width 2.42
misspecified   coverage 0.80  median |rel err| 0.51  mean interval width 2.69


## What the model recovered about the transforms

The misspecified arm cannot represent a delayed peak, so whatever the true delay
was has to be absorbed somewhere else — usually into the decay rate and the
coefficient. This is where that shows.

In [6]:
print(json.dumps(mmm["fits"]["misspecified"]["recovered_parameters"], indent=2)[:1800])

{
  "display_prog": {
    "adstock_params": {
      "alpha": 0.21785425304242845
    },
    "saturation_params": {
      "beta": 0.054820056064621425,
      "lam": 1.8345684891660534
    }
  },
  "search_brand": {
    "adstock_params": {
      "alpha": 0.22686802856041888
    },
    "saturation_params": {
      "beta": 0.041363509424660515,
      "lam": 1.8710150280832836
    }
  },
  "search_nonbrand": {
    "adstock_params": {
      "alpha": 0.22307429987866084
    },
    "saturation_params": {
      "beta": 0.029177054458859337,
      "lam": 1.8461522199380367
    }
  },
  "social_paid": {
    "adstock_params": {
      "alpha": 0.24690232313453456
    },
    "saturation_params": {
      "beta": 0.02972172605683234,
      "lam": 1.8853077167470906
    }
  },
  "video_ctv": {
    "adstock_params": {
      "alpha": 0.33089187520724783
    },
    "saturation_params": {
      "beta": 0.07084797437408871,
      "lam": 2.037946379661954
    }
  }
}
